# Numerical accuracy of a joint DES Y3 Fisher calculation

This notebook asks whether changes in numerical resolution change the derivatives that determine the marginalized $w_0$–$w_a$ ellipse. It starts from a **full Fisher calculation at the declared default accuracy**. It then reads less expensive tests at selected parameter columns, followed by full recalculations where they have been completed. NLA and TATT retain separate baselines, priors, parameter sets and diagnostic weights.


The initial TATT forecast was rejected because interpolation generated an artificial high-multipole contribution. The corrected calculation uses signed component ratios and revised triangle and angular grids. The [interpolation study](tests/studies/interpolation_validation.md) records the independent checks. The diagnostics below test the configured numerical calculation; they do not validate the remaining physical approximations or establish a published cosmological-information gain.
These are numerical checks of a demonstration forecast. The input covariance, source preparation, unvalidated agreement with the exact measured aperture estimator, scale cuts and physical approximations are recorded in the YAML files. Passing a numerical check would not establish a reproduction of either published DES analysis.

## Contents

1. [Set up the notebook](#setup)
2. [Why the two-point quadrature was changed](#xi)
3. [Define the baseline and the small follow-up calculations](#baseline)
4. [Understand the noise estimate and its limitations](#noise)
5. [Inspect the selected columns and cheap checks](#screen)
6. [Inspect completed full derivative checks](#full)
7. [Run additional checks explicitly](#run)
8. [Decide what the evidence establishes](#interpretation)

<a name="setup"></a>
## 1. Step :one: — Set up the notebook

Run from the `des_3pcf` project directory. BLAS uses one thread throughout; OpenMP owns any parallel C work and is capped at four workers. Loading saved arrays and drawing figures does not launch a model evaluation. The public `threadpoolctl` API also sets already loaded BLAS libraries to one thread, which matters when this notebook shares a previously used kernel.

The plotting functions receive arrays and leave fonts to the notebook. They are shared with other projects through `cosmolike_notebook_utils`; the project wrapper checks saved-file identities and decides which results are available.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import importlib
import os
from pathlib import Path
import sys

for variable in (
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[variable] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_THREAD_LIMIT"] = "4"

PROJECT = Path.cwd().resolve()
if not (PROJECT / "PAPER_TARGETS.md").is_file():
    raise RuntimeError("Run this notebook from Cocoa/projects/des_3pcf.")
CORE = PROJECT.parents[1] / "external_modules/code/cosmolike_core"
sys.path.insert(0, str(PROJECT / "interface"))
sys.path.insert(0, str(CORE))

import matplotlib
from matplotlib import pyplot as plt
from IPython.display import Markdown, display
from threadpoolctl import threadpool_info, threadpool_limits

blas_limit = threadpool_limits(limits=1, user_api="blas")
for pool in threadpool_info():
    if pool["user_api"] == "blas" and pool["num_threads"] != 1:
        raise RuntimeError("A loaded BLAS pool still has more than one thread.")

from cosmolike_des_3pcf_accuracy_wrappers import AccuracyStudy, plot_xi_preflight

# Figure style of the whole notebook: the shared plotting functions never change
# rcParams, so these lines set every font, tick and grid default.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'

<a name="xi"></a>
## 2. Step :two: — Inspect the fiducial two-point quadrature

A small error in a mean vector can become a large derivative error after division by a small parameter step. We therefore first inspected the numerical integral producing $\xi_\pm$, independently of the later Fisher diagnostics. The saved check varies the number of Fourier quadrature nodes $N_\ell$, the number of nodes used to average each angular bin $N_\theta$, and the integration upper bound $\ell_{\max}$.

The plotted quantity is
\[
D_\xi^2 =
(\boldsymbol{\xi}_{q}-\boldsymbol{\xi}_{\rm ref})^{\mathsf T}
C_\xi^{-1}
(\boldsymbol{\xi}_{q}-\boldsymbol{\xi}_{\rm ref}),
\]
where $q$ labels a numerical setting and $C_\xi$ is the supplied two-point covariance block. This is a covariance-weighted distance between predictions, not a fit to observed data. The reference uses $32768$ Fourier nodes, $256$ angular nodes and $\ell_{\max}=80000$.

This historical check used the previous `linear_midpoint` source reconstruction
and `runs/y3_demo_physics.npz`. It explains the quadrature change; its
quoted distances are not an accuracy certificate for the current
measured-edge PCHIP sources. The new `xi_ell_2` scan below checks
derivatives with the current inputs. Angular averaging and the integration
upper bound remain separate controls.

The historical check found $D_\xi^2\simeq14.9$ with $256/32/40000$ and about $0.00126$ with $4096/32/40000$. The latter became the default used for the Fisher bases. The plateau reached by increasing $N_\ell$ alone is instructive: angular averaging and the finite integration bound still leave differences. The reference has zero distance from itself and is omitted from the logarithmic plot without substituting a floor. This check covers the fiducial vector; displaced parameter vectors and their derivatives require the following tests.

In [ ]:
plot_xi_preflight(directory=PROJECT / "runs/xi_refinement");

<a name="baseline"></a>
## 3. Step :three: — Name the baselines and follow-up plan

For the uncompressed joint vector $\boldsymbol d=(\xi_+,\xi_-,\langle M_{\rm ap}^3\rangle)$, the project forms
\[
F_{\alpha\beta}
= D_\alpha^{\mathsf T}C^{-1}D_\beta
+F^{\rm prior}_{\alpha\beta},
\qquad
D_\alpha=\frac{\partial\boldsymbol d}{\partial p_\alpha}.
\]
The covariance $C$ is held fixed. Every stated Gaussian prior is included before marginalization. If $S=F^{-1}$ is the parameter covariance, the figure of merit is
\[
{\rm FoM}_{w_0w_a} =
\left[\det S_{\{w_0,w_a\}}\right]^{-1/2}.
\]
Thus a nuisance derivative can alter the dark-energy ellipse even when that nuisance is not one of its two displayed axes.

The full baseline stores five mean vectors per parameter at spacings $h$ and $h/2$. Several points are shared, so both derivative sets require six distinct off-center means per parameter and one shared fiducial mean. The cheap scan reuses these arrays and this full Fisher matrix. Only the selected parameter columns receive new mean evaluations at a candidate accuracy. The notebook reads the completed files; it does not launch missing work during a status check.

In [ ]:
studies = {
    "NLA": AccuracyStudy(
        configuration=PROJECT / "configs/des_y3_nla_demo_fisher.yaml",
        output=PROJECT / "runs/nla_accuracy",
        plan=PROJECT / "configs/des_y3_fisher_accuracy.yaml",
    ),
    "TATT": AccuracyStudy(
        configuration=PROJECT / "configs/des_y3_tatt_demo_fisher.yaml",
        output=PROJECT / "runs/tatt_accuracy",
        plan=PROJECT / "configs/des_y3_fisher_accuracy.yaml",
    ),
}
for label, study in studies.items():
    display(Markdown(f"### {label}\n\n" + study.summary_markdown()))

<a name="noise"></a>
## 4. Understand what the inexpensive estimator measures

Write $\boldsymbol d_j=\boldsymbol d(p_\alpha+j\epsilon_\alpha)$, holding all other parameters fixed. Here $\epsilon_\alpha$ is an **absolute** step in the parameter's units, including for negative or zero fiducial values. The fourth difference and the fourth-order central derivative are
\[
\boldsymbol q_\alpha
=\boldsymbol d_{-2}-4\boldsymbol d_{-1}
+6\boldsymbol d_0-4\boldsymbol d_1+\boldsymbol d_2,
\qquad
\widehat D_\alpha
=\frac{\boldsymbol d_{-2}-8\boldsymbol d_{-1}
+8\boldsymbol d_1-\boldsymbol d_2}{12\epsilon_\alpha}.
\]
The fourth difference cancels smooth polynomials through cubic order. If the remaining error behaves as independent, equally distributed jitter at the five points, the squared difference coefficients sum to $70$. The shared estimator therefore uses
\[
\widehat\sigma_\alpha^2
=\frac{\boldsymbol q_\alpha^{\mathsf T}C^{-1}\boldsymbol q_\alpha}{70},
\qquad
a_\alpha=\frac{130}{144}
\frac{\widehat\sigma_\alpha^2}{\epsilon_\alpha^2}.
\]
The factor $130/144$ is the squared coefficient sum of the derivative stencil. In that noise model, $a_\alpha$ is an expected diagonal addition to parameter information, with units of inverse parameter squared.

The Fisher machinery note's `eq:fomall` relates these additions to the ellipse. Let $T=\{w_0,w_a\}$, let $s_\alpha^2=S_{\alpha\alpha}$, and define
\[
R_\alpha^2 =
\frac{S_{\alpha T}S_{TT}^{-1}S_{T\alpha}}{S_{\alpha\alpha}}.
\]
Then the first-order expectation is
\[
\delta\ln{\rm FoM}\simeq
\frac12\sum_\alpha R_\alpha^2s_\alpha^2a_\alpha.
\]
The shared helper also rescales the FoM by calculating the exact ratio for $F+\operatorname{diag}(a)$, without truncating this matrix calculation at first order. Its exactness concerns the **assumed diagonal-noise model**. It does not make that model an exact description of the numerical error.

This distinction determines the follow-up tests. A smooth fourth derivative can contribute to $\boldsymbol q_\alpha$ without being numerical noise. A coherent error proportional to the parameter displacement can change $\widehat D_\alpha$ while canceling from $\boldsymbol q_\alpha$. Correlations between stencil points or between parameter columns also invalidate the independent-jitter coefficients. The saved $h$ and $h/2$ checks share points and are not independent measurements to average. Actual derivative changes and full recalculations remain necessary.

<a name="screen"></a>
## 5. Step :four: — Inspect all baseline columns, then selected candidates

The first figure shows each parameter's measured contribution $R_\alpha^2s_\alpha^2a_\alpha$ as a percentage of their sum at default accuracy. The scan always includes the two FoM parameters and then selects columns with the largest measured contributions, up to the declared cap. The printed coverage is the achieved baseline share, which may fall below the requested $95\%$. A small correlation weight alone gives no bound on an omitted column's unknown numerical noise.

The next figure compares two different diagnostics. The blue curve uses the shared diagonal-noise estimate with **fixed baseline weights**, normalized by the estimate from the same selected baseline columns. It is a diagnostic of expected information from jitter, not an exact candidate-to-baseline FoM prediction or a correction to subtract from the baseline. The orange curve replaces the selected derivative columns by their actually calculated candidate values and leaves the other columns at baseline. It detects some coherent changes that the fourth difference misses, but still is a **partial replacement**, not a full refined Fisher.

All differences are displayed as $100(r-1)$ percent. A zero reference line means agreement with the fixed baseline in the stated diagnostic. Missing or rank-unresolved results remain gaps; no acceptance threshold is inferred.

`multipole_1p5` refines the interpolation tables in Fourier size and
shape angle. It does not raise the retained angular orders: all planned
cases keep $L_{\max}=M_{\max}=20$. Convergence with respect to those
orders requires a separate truncation study.


In [ ]:
display(Markdown("### NLA: measured baseline contributions"))
studies["NLA"].plot_noise_contributions();

In [ ]:
display(Markdown("### NLA: selected-column accuracy checks"))
studies["NLA"].plot_accuracy_screen();

In [ ]:
display(Markdown("### TATT: measured baseline contributions"))
studies["TATT"].plot_noise_contributions();

In [ ]:
display(Markdown("### TATT: selected-column accuracy checks"))
studies["TATT"].plot_accuracy_screen();

<a name="full"></a>
## 6. Step :five: — Inspect completed full derivative checks

These figures recalculate the FoM from every derivative column in each available full archive. Both the $h$ and $h/2$ results use the same default-accuracy FoM denominator chosen by the scan. Their separation tests parameter-step sensitivity; changes between numerical settings test integration and interpolation sensitivity. No parameter covariance is obtained from a pseudoinverse if the matrix fails the project's rank test.

The loader checks the configuration digest, parameter order, full data-row order, fixed covariance and archived physical-input/code identities before combining files. Only completed full archives enter the figure. Candidate scans without a full recalculation are listed as pending. If only the baseline appears, the figure establishes no completed accuracy refinement, even when the cheap diagnostic was small. These numerical FoMs are diagnostic quantities, not accepted survey forecasts.

Historical figures compare the archived identities with one another.
An explicit new full check also verifies the current input and code files
before launching; changed implementations cannot masquerade as a
resolution-only comparison.


In [ ]:
display(Markdown("### NLA: available full derivative recalculations"))
studies["NLA"].plot_full_checks();

In [ ]:
display(Markdown("### TATT: available full derivative recalculations"))
studies["TATT"].plot_full_checks();

<a name="run"></a>
## 7. Run additional checks explicitly

The methods below launch real calculations in separate processes with BLAS fixed to one thread and at most four OpenMP workers. Run them only when the machine is available. They are intentionally shown as explicit commands, rather than a Boolean flag that makes an apparently executed notebook quietly skip the calculation.

A selected first pass can isolate the two-point tables, Fourier quadrature,
triangle interpolation and line-of-sight integration:

```python
studies["NLA"].run_scan(
    only=["core_2pt_1p5", "xi_ell_2", "triangle_1p5", "chi_2"],
)
```

Existing matching cases are retained; every uncomputed candidate remains
listed as pending. To complete all planned scans after the baseline exists:

```python
studies["TATT"].run_scan()
```

Run the NLA scan separately:

```python
studies["NLA"].run_scan()
```

To calculate one selected candidate, use its exact plan name. For example, once the saved `chi_2` scan is available:

```python
studies["TATT"].run_full_check(case="chi_2")
```

A full check evaluates every derivative at both step lengths through the ordinary Fisher driver. Re-run the loading and plotting cells afterward; they reread the files without requiring a kernel restart. Existing outputs are not silently overwritten. The printed command below is also usable outside the notebook.

In [ ]:
for label, study in studies.items():
    print(label + ":")
    print(study.scan_command())

<a name="interpretation"></a>
## 8. Read the evidence at its stated scope

A stable noise estimate is useful for deciding where to spend further calculations. A stable full FoM across numerical settings and parameter steps is stronger evidence for that declared numerical problem. Neither test changes the physical problem being solved. Keep the following separate:

- **Sampling and quadrature:** number of line-of-sight, interpolation, transform, multipole and angular nodes. The scan changes their resolution while keeping the model fixed.
- **Support and estimator:** integration bounds, finite SAS side support, bin weighting, angular averaging and aperture definition. More nodes at fixed bounds do not test these choices.
- **Spin and ordered legs:** Fourier phases, natural-component convention and ordered TATT source weights. A common convention error can survive every resolution test.
- **Physical assumptions:** IA power prescription, baryonic response, effective-growth treatment of massive neutrinos, scale cuts and covariance-model consistency. These need physical comparisons or separate model variations.
- **Forecast design:** priors, fitted parameter set, fixed-covariance assumption and the absence of compression. The saved configuration specifies them; they must not change unnoticed during an accuracy comparison.

The implementation uses the existing [shared Fisher diagnostic](../../external_modules/code/cosmolike_core/cosmolike_notebook_utils/fisher.py), with project adapters in [fisher_accuracy.py](python/des3pcf/fisher_accuracy.py). Its derivation is in the [Fisher machinery note](../lssty1xdesi/notes/fisher_machinery/fisher_machinery.tex), particularly `eq:fourthdiff`, `eq:noisehat`, `eq:derivnoise`, `eq:spuriousprior` and `eq:fomall`. The scientific scope of the DES configurations is recorded in [PAPER_TARGETS.md](PAPER_TARGETS.md) and the YAML notes.